# Let Science Data Speak — Book 1, Companion Notebook — Chapter 5
## A first analysis of journal articles with OpenAlex

This notebook goes with **Chapter 5 (Journal articles)** of *Let Science Data Speak, Book 1*. It follows the same six steps as the chapter's first analysis:

1. Define the question and the search
2. Collect (with a search log)
3. Clean
4. Add one column your question needs (country of the first author)
5. Count and chart
6. Interpret

You only need to change the **Settings** cell. Then choose *Run all*. It runs in Google Colab, Jupyter or VS Code, and needs no account.

*Licence: CC BY 4.0. Data: [OpenAlex](https://openalex.org) (CC0).*

### 1. Settings — define your question and your search

Write your question in one sentence. Then build a search string: synonyms joined by `OR` inside a concept, and concepts joined by `AND`. OpenAlex searches titles, abstracts and full text where it has it.

`REGION` is the list of countries for your equity question, as two-letter codes (for example `CM` for Cameroon, `CA` for Canada).

In [ ]:
QUESTION = "How has research on phage therapy grown, who leads it, and how much is led from Africa?"

SEARCH = '("phage therapy" OR "bacteriophage therapy" OR "phage treatment")'

YEAR_FROM, YEAR_TO = 2000, 2025      # avoid the current year: it is always incomplete
DOC_TYPES = ["article", "review", "preprint"]
MAX_RECORDS = 2000                   # records to download for the detailed steps (counts per year use all results)

REGION_NAME = "Africa"
REGION = ["DZ","AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","DJ","EG","GQ","ER","SZ","ET",
          "GA","GM","GH","GN","GW","KE","LS","LR","LY","MG","MW","ML","MR","MU","MA","MZ","NA","NE","NG","RW",
          "ST","SN","SC","SL","SO","ZA","SS","SD","TZ","TG","TN","UG","ZM","ZW"]

### Optional: an OpenAlex API key

Without a key, OpenAlex gives a small free daily budget shared by everyone on your network. A free personal key gives you your own budget. Get one at [openalex.org/settings/api](https://openalex.org/settings/api) and store it as an environment variable named `OPENALEX_API_KEY` (in Colab: the key icon, *Secrets*). **Never paste a key into the notebook itself**, especially if you share it.

In [ ]:
import os, time, json, random, datetime
import requests
import pandas as pd
import matplotlib.pyplot as plt

API = "https://api.openalex.org/works"
KEY = os.environ.get("OPENALEX_API_KEY")          # optional, read from the environment only

def get(params):
    params = dict(params)
    if KEY:
        params["api_key"] = KEY
    for attempt in range(5):
        r = requests.get(API, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        time.sleep(2 ** attempt)                   # wait and retry on errors or rate limits
    r.raise_for_status()

FILTER = f"publication_year:{YEAR_FROM}-{YEAR_TO},type:{'|'.join(DOC_TYPES)}"
print("Question:", QUESTION)
print("Filter:  ", FILTER)

### 2. Collect — with a search log

First, OpenAlex counts **all** matching works per year (fast and complete). Then the notebook downloads up to `MAX_RECORDS` records for the detailed steps. These come in OpenAlex's relevance order, so if your topic has more works than `MAX_RECORDS`, the sample favours the most relevant ones; raise `MAX_RECORDS` to get them all. Every run is written to `search_log.csv`.

In [ ]:
# Complete counts per year
g = get({"search": SEARCH, "filter": FILTER, "group_by": "publication_year"})
per_year = (pd.DataFrame(g["group_by"])
              .rename(columns={"key": "year", "count": "works"})
              .astype({"year": int})
              .sort_values("year"))
total = int(per_year["works"].sum())
print(f"{total:,} works match the search")

# Detailed records: cursor paging with a key, page numbers without one (OpenAlex limit: 10,000 records)
rows, fields = [], "id,doi,title,publication_year,type,authorships,primary_location,open_access,cited_by_count"
base = {"search": SEARCH, "filter": FILTER, "per-page": 200, "select": fields}
cursor, page_no = "*", 1
try:
    while len(rows) < min(MAX_RECORDS, 10000):
        p = dict(base, cursor=cursor) if KEY else dict(base, page=page_no)
        res = get(p)
        if not res["results"]:
            break
        rows += res["results"]
        cursor, page_no = res["meta"].get("next_cursor"), page_no + 1
        if KEY and not cursor:
            break
except requests.HTTPError as e:
    print(f"Stopped early ({e.response.status_code}): the daily budget may be used up. "
          "Add a free API key (see above) or try again tomorrow. Continuing with the records downloaded so far.")
rows = rows[:MAX_RECORDS]
print(f"{len(rows):,} records downloaded")

# Search log: date, database, search, filters, number of results
log = pd.DataFrame([{
    "date": datetime.date.today().isoformat(), "database": "OpenAlex", "search": SEARCH,
    "filters": FILTER, "results_total": total, "records_downloaded": len(rows)}])
log.to_csv("search_log.csv", mode="a", index=False, header=not os.path.exists("search_log.csv"))
log

### 3. Clean

Flatten the records, remove duplicates (same DOI, or same title and year), and draw **50 random records to check by hand**. Open `check_by_hand.csv`, mark each record *relevant* or *not*, and refine the search in the Settings cell if too many are off-topic.

In [ ]:
def first_author(w):
    for a in w.get("authorships") or []:
        if a.get("author_position") == "first":
            return a
    return (w.get("authorships") or [None])[0]

def row(w):
    fa = first_author(w) or {}
    src = ((w.get("primary_location") or {}).get("source") or {})
    return {
        "openalex_id": w["id"],
        "doi": (w.get("doi") or "").lower().replace("https://doi.org/", ""),
        "title": w.get("title") or "",
        "year": w.get("publication_year"),
        "type": w.get("type"),
        "journal": src.get("display_name"),
        "first_author_countries": "|".join(fa.get("countries") or []),
        "is_oa": (w.get("open_access") or {}).get("is_oa"),
        "cited_by": w.get("cited_by_count"),
    }

df = pd.DataFrame([row(w) for w in rows])
n0 = len(df)
df = df[~(df["doi"].ne("") & df.duplicated("doi"))]
df = df[~df.assign(t=df["title"].str.lower().str.strip()).duplicated(["t", "year"])]
print(f"Removed {n0 - len(df)} duplicates; {len(df):,} records left")

df.sample(min(50, len(df)), random_state=1).assign(relevant="").to_csv("check_by_hand.csv", index=False)
df.head()

### 4. Add one column your question needs

For an equity question: **is the first author based in the region you study?** Records with no affiliation country stay *unknown*: say how many in your write-up.

In [ ]:
def in_region(c):
    if not c:
        return "unknown"
    return "yes" if any(x in REGION for x in c.split("|")) else "no"

df["first_author_in_region"] = df["first_author_countries"].map(in_region)
df["first_author_in_region"].value_counts()

### 5. Count and chart

Four results, as in the chapter: articles per year, top ten countries, top ten journals, and the share of articles with a first author in your region.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(per_year["year"], per_year["works"], color="#3b6ea5")
ax.set_title("Works per year (all matching records)")
ax.set_xlabel("Publication year"); ax.set_ylabel("Works")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

countries = (df["first_author_countries"].str.split("|").explode()
               .replace("", pd.NA).dropna().value_counts().head(10))
journals = df["journal"].dropna().value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
countries.sort_values().plot.barh(ax=axes[0], color="#3b6ea5", title="Top 10 first-author countries")
journals.sort_values().plot.barh(ax=axes[1], color="#3b6ea5", title="Top 10 journals")
for a in axes:
    a.set_xlabel("Records (downloaded sample)"); a.set_ylabel(""); a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

known = df[df["first_author_in_region"] != "unknown"]
share = (known["first_author_in_region"] == "yes").mean()
print(f"First author based in {REGION_NAME}: {share:.1%} of {len(known):,} records with a known country "
      f"({(df['first_author_in_region'] == 'unknown').sum():,} records have no country)")

In [ ]:
df.to_csv("articles_clean.csv", index=False)
per_year.to_csv("works_per_year.csv", index=False)
print("Saved: articles_clean.csv, works_per_year.csv, check_by_hand.csv, search_log.csv")

### 6. Interpret

For each result, ask the chapter's three questions:

- **Is it real, or caused by the database?** OpenAlex coverage grows over time, the last year is never complete, and affiliation countries are missing for part of the records.
- **What could explain it?**
- **What decision does it support?**

Then write three sentences: **what is growing, who leads, and what seems to be missing.**

*Remember what this source gets wrong (Chapter 5): one database is never all of science, search terms shape the result, and counts show where to look, not what to conclude. Check a sample by hand, and compare important counts with a second source such as PubMed.*

Keep `articles_clean.csv`. In Chapters 6 and 7 you will add theses and citations to it.